# AutoGaze External CUDA Verification

This notebook verifies the pushed AutoGaze reproduction branch on a CUDA runtime. It is intended for Kaggle or Colab after GPU access is enabled.


In [ ]:
import json, os, pathlib, subprocess, sys, textwrap, time
import torch

print('python:', sys.version)
print('torch:', torch.__version__)
print('cuda_available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required. Enable a Kaggle/Colab GPU runtime before continuing.')
print('cuda_device:', torch.cuda.get_device_name(0))


In [ ]:
BRANCH = 'codex/autogaze-repro'
REPO_URL = 'https://github.com/manricheon/AutoGaze.git'
WORK_ROOT = pathlib.Path('/kaggle/working')
OUTPUT_ROOT = pathlib.Path('/kaggle/working/autogaze_vjepa_outputs')
WEIGHTS_ROOT = pathlib.Path('/kaggle/working/autogaze_weights')
REPORT_PATH = pathlib.Path('/kaggle/working/autogaze_vjepa_outputs/colab_verification.md')
REPO_DIR = WORK_ROOT / 'AutoGaze'
RUN_VJEPA_QWEN = True
RUN_NVILA_SINGLE = True
RUN_NVILA_HLVID_MINI = True
RUN_QWEN_PLUGIN_HLVID_MINI = True
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
WEIGHTS_ROOT.mkdir(parents=True, exist_ok=True)

def run(cmd, *, cwd=None):
    cmd = [str(x) for x in cmd]
    print('\n$', ' '.join(cmd))
    subprocess.check_call(cmd, cwd=str(cwd) if cwd else None)

if not REPO_DIR.exists():
    run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR])
else:
    run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR)
    run(['git', 'checkout', BRANCH], cwd=REPO_DIR)
    run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR)
os.chdir(REPO_DIR)
run(['git', 'log', '--oneline', '-1'])


In [ ]:
run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-repro.txt', 'transformers>=4.57.0', 'qwen-vl-utils', 'av', 'pytest'])


In [ ]:
run([
    sys.executable, 'scripts/verify_autogaze_entrypoints.py',
    '--output-json', OUTPUT_ROOT / 'entrypoint_verification.json',
    '--output-md', OUTPUT_ROOT / 'entrypoint_verification.md',
])


In [ ]:
if RUN_VJEPA_QWEN:
    run([
        sys.executable, 'scripts/run_colab_autogaze_cuda_smoke.py',
        '--weights-root', WEIGHTS_ROOT,
        '--output-root', OUTPUT_ROOT,
        '--video', 'inputs/hlvid_example/clip_av_video_5_001.mp4',
        '--num-video-frames', '16',
        '--frames-per-clip', '16',
        '--video-resize-longest-edge', '224',
        '--max-new-tokens', '4',
    ])


In [ ]:
if RUN_VJEPA_QWEN:
    run([
        sys.executable, '-m', 'repro.colab_verification_report',
        '--output-md', REPORT_PATH,
        '--title', 'AutoGaze External CUDA Verification',
        '--video', 'inputs/hlvid_example/clip_av_video_5_001.mp4',
        '--query', 'Describe the video in one short sentence.',
        '--entrypoint-verification-json', OUTPUT_ROOT / 'entrypoint_verification.json',
        '--case', f"vjepa_qwen_dense_off={OUTPUT_ROOT / 'vjepa_qwen_dense_off_cuda_smoke.json'}",
        '--case', f"autogaze_vjepa_qwen_on={OUTPUT_ROOT / 'autogaze_vjepa_qwen_on_cuda_smoke.json'}",
    ])


In [ ]:
if RUN_VJEPA_QWEN:
    summary_path = OUTPUT_ROOT / 'colab_autogaze_cuda_smoke_summary.json'
    report_path = REPORT_PATH
    summary = json.loads(summary_path.read_text())
    print(json.dumps(summary['summary'], indent=2))
    print('report:', report_path)
    print('visualizations:', OUTPUT_ROOT / 'visualizations')
    assert summary['summary']['passed'] is True
    dense = summary['results']['vjepa_qwen_dense_off']
    ag = summary['results']['autogaze_vjepa_qwen_on']
    assert dense['status'] == 'passed'
    assert ag['status'] == 'passed'
    assert ag['tokens']['vjepa_selected_tokens'] < ag['tokens']['vjepa_raw_tokens']
    assert ag['tokens']['qwen_visual_tokens_inserted'] == ag['tokens']['vjepa_selected_tokens']


In [ ]:
if RUN_NVILA_SINGLE:
    os.environ['PYTHONPATH'] = str(REPO_DIR) + os.pathsep + os.environ.get('PYTHONPATH', '')
    os.environ.setdefault('HF_HOME', str(WORK_ROOT / 'hf_cache_nvila'))
    os.environ.setdefault('TRANSFORMERS_CACHE', str(WORK_ROOT / 'hf_cache_nvila' / 'transformers'))
    os.environ.setdefault('HF_HUB_CACHE', str(WORK_ROOT / 'hf_cache_nvila' / 'hub'))
    nvila_out = OUTPUT_ROOT / 'nvila_single_smoke'
    nvila_out.mkdir(parents=True, exist_ok=True)
    nvila_base = [
        sys.executable, '-m', 'repro.nvila_runner',
        '--mode', 'single',
        '--model-path', 'nvidia/NVILA-8B-HD-Video',
        '--autogaze-model', WEIGHTS_ROOT / 'nvidia__AutoGaze',
        '--device', 'cuda', '--device-map', 'auto', '--dtype', 'float16',
        '--video', 'inputs/hlvid_example/clip_av_video_5_001.mp4',
        '--prompt', 'Describe the video in one short sentence.',
        '--num-video-frames', '16', '--num-video-frames-thumbnail', '16',
        '--max-tiles-video', '1',
        '--video-resize-longest-edge', '224', '--video-decode-strategy', 'seek',
        '--max-batch-size-autogaze', '2', '--max-batch-size-siglip', '1',
        '--max-new-tokens', '1',
    ]
    run([*nvila_base, '--gazing-mode', 'keep-all-single', '--output-json', nvila_out / 'keep_all_single.json'], cwd=REPO_DIR)
    run([*nvila_base, '--gazing-mode', 'autogaze', '--output-json', nvila_out / 'autogaze.json'], cwd=REPO_DIR)
    keep = json.loads((nvila_out / 'keep_all_single.json').read_text())
    ag_nvila = json.loads((nvila_out / 'autogaze.json').read_text())
    print('nvila_single_keep_answer:', (keep.get('summary') or {}).get('answer'))
    print('nvila_single_autogaze_answer:', (ag_nvila.get('summary') or {}).get('answer'))
    print('nvila_single_outputs:', nvila_out)


In [ ]:
if RUN_NVILA_HLVID_MINI:
    dataset = OUTPUT_ROOT / 'hlvid_mini_dataset'
    nvila_hlvid_out = OUTPUT_ROOT / 'nvila_hlvid_mini'
    dataset.mkdir(parents=True, exist_ok=True)
    nvila_hlvid_out.mkdir(parents=True, exist_ok=True)
    manifest = dataset / 'manifest.jsonl'
    manifest.write_text(json.dumps({
        'question_id': 'kaggle_smoke_001',
        'category': 'smoke',
        'video_path': 'clip_av_video_5_001.mp4',
        'question': 'What does the white text on the green road sign say?\nA. Hampden St\nB. Hampden Ave\nC. HampdenBlvd\nD. Hampden Rd\nPlease answer directly with the letter of the correct answer.',
        'answer': 'B',
    }) + '\n', encoding='utf-8')
    run([
        sys.executable, 'scripts/run_hlvid_folder_benchmark.py',
        '--dataset-dir', dataset,
        '--manifest', manifest,
        '--video-root', REPO_DIR / 'inputs/hlvid_example',
        '--output-dir', nvila_hlvid_out,
        '--model-path', 'nvidia/NVILA-8B-HD-Video',
        '--autogaze-model', WEIGHTS_ROOT / 'nvidia__AutoGaze',
        '--device', 'cuda', '--device-map', 'auto', '--dtype', 'float16',
        '--num-video-frames', '16', '--num-video-frames-thumbnail', '16',
        '--max-tiles-video', '1',
        '--video-resize-longest-edge', '224', '--video-decode-strategy', 'seek',
        '--max-batch-size-autogaze', '2', '--max-batch-size-siglip', '1',
        '--max-new-tokens', '1',
        '--limit', '1', '--continue-on-error', '--skip-keep-all',
    ], cwd=REPO_DIR)
    report = nvila_hlvid_out / 'hlvid_autogaze_gain_report.json'
    assert report.exists(), report
    payload = json.loads(report.read_text())
    print('nvila_hlvid_mini_report:', report)
    print('nvila_hlvid_mini_outputs:', sorted(path.name for path in nvila_hlvid_out.glob('*')))


In [ ]:
if RUN_QWEN_PLUGIN_HLVID_MINI:
    dataset = OUTPUT_ROOT / 'hlvid_mini_dataset'
    manifest = dataset / 'manifest.jsonl'
    if not manifest.exists():
        dataset.mkdir(parents=True, exist_ok=True)
        manifest.write_text(json.dumps({
            'question_id': 'kaggle_smoke_001',
            'category': 'smoke',
            'video_path': 'clip_av_video_5_001.mp4',
            'question': 'What does the white text on the green road sign say?\nA. Hampden St\nB. Hampden Ave\nC. HampdenBlvd\nD. Hampden Rd\nPlease answer directly with the letter of the correct answer.',
            'answer': 'B',
        }) + '\n', encoding='utf-8')
    qwen_hlvid_out = OUTPUT_ROOT / 'qwen_plugin_hlvid_mini'
    qwen_hlvid_out.mkdir(parents=True, exist_ok=True)
    run([
        sys.executable, 'scripts/run_hlvid_folder_benchmark.py',
        '--plugin-suite', 'qwen',
        '--dataset-dir', dataset,
        '--manifest', manifest,
        '--video-root', REPO_DIR / 'inputs/hlvid_example',
        '--output-dir', qwen_hlvid_out,
        '--plugin-model', f"qwen3-vl={WEIGHTS_ROOT / 'Qwen__Qwen2.5-VL-3B-Instruct'}",
        '--autogaze-model', WEIGHTS_ROOT / 'nvidia__AutoGaze',
        '--device', 'cuda', '--device-map', 'auto', '--dtype', 'float16',
        '--attn-implementation', 'eager',
        '--num-video-frames', '16', '--num-video-frames-thumbnail', '0',
        '--max-tiles-video', '1',
        '--video-resize-longest-edge', '224', '--video-decode-strategy', 'seek',
        '--qwen-vit-chunk-frames', '16', '--qwen-vit-max-spatial-chunks', '1',
        '--autogaze-target-scales', '64+128+192+224',
        '--autogaze-target-patch-size', '16', '--autogaze-tile-size', '224',
        '--max-batch-size-autogaze', '2',
        '--max-new-tokens', '1',
        '--limit', '1', '--continue-on-error',
    ], cwd=REPO_DIR)
    qwen_runs = sorted((qwen_hlvid_out / 'runs').glob('*/*.json'))
    assert qwen_runs, qwen_hlvid_out
    print('qwen_plugin_hlvid_mini_outputs:', qwen_hlvid_out)
    for path in qwen_runs:
        payload = json.loads(path.read_text())
        generation = payload.get('generation') or {}
        metrics = generation.get('metrics') or {}
        tokens = metrics.get('tokens') or {}
        print(path.parent.name, payload.get('implementation_status'), generation.get('status'), generation.get('text'), tokens.get('visual_tokens_before_prune'), tokens.get('visual_tokens_after_prune'))
